[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/Trace-Bench/blob/main/notebooks/04_gradio_ui.ipynb)

# Trace-Bench — UI Launch + Browse + Resume + TensorBoard/MLflow

This notebook demonstrates the UI behavior:
- Launch a small stub run (2x2 matrix) to produce run artifacts
- Start the Gradio UI with `trace-bench ui`
- Browse runs, filter results, inspect jobs
- Resume a previous run using the UI resume flow
- TensorBoard and MLflow integration (optional)

**Artifacts under `runs/<run_id>/` are canonical.** The UI reads filesystem; MLflow/TB mirror it.

## Expected Outputs

- Gradio UI launches at localhost:7860 (public URL via `--share` on Colab).
- **Launch Run tab**: config selection, provider dropdown, run button.
- **Browse Runs tab**: completed runs list, results table, leaderboard.
- **Job Inspector tab**: job metadata, events, state snapshots.

In [ ]:
from datetime import date
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

def bench_dir(project="bench", sub="trace_bench_ui", local="/content/bench"):
    drive_root = Path("/content/drive/MyDrive")
    root = drive_root if drive_root.is_dir() else Path(local)
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)

In [ ]:
# Clone repos + install deps
!git clone --depth 1 https://github.com/AgentOpt/Trace-Bench.git /content/Trace-Bench 2>/dev/null || (cd /content/Trace-Bench && git pull --ff-only)
!git clone --depth 1 --branch experimental https://github.com/AgentOpt/OpenTrace.git /content/OpenTrace 2>/dev/null || (cd /content/OpenTrace && git fetch origin experimental && git checkout experimental && git pull --ff-only)

%cd /content/Trace-Bench
!python -m pip install -q pyyaml gradio "litellm==1.75.0" "aiohttp>=3.9,<3.13"

# OpenTrace currently declares python>=3.13 in pyproject; on Colab (3.12) use source checkout via PYTHONPATH.
!PYTHONPATH=/content/OpenTrace:/content/Trace-Bench/src:$PYTHONPATH python -c "import opto, trace_bench, litellm; print('imports ok')"


> **Note:** These notebooks use PYTHONPATH to import from `src/` and typically do not require a runtime restart. If you hit import errors after installs, restart once and rerun the cell.

In [ ]:
%%bash
cd /content/Trace-Bench

# Use the bundled demo config
PYTHONPATH=/content/OpenTrace:/content/Trace-Bench/src:$PYTHONPATH python -m trace_bench run --config configs/m3_ui_demo.yaml --runs-dir "$RUNS_DIR"

echo ""
echo "--- Run artifacts ---"
ls -la "$RUNS_DIR"/*/
echo ""
echo "--- results.csv (head) ---"
head -5 "$RUNS_DIR"/*/results.csv 2>/dev/null || echo "(no results.csv found)"

In [4]:
# Show run summary
import json, glob

for summary_path in sorted(glob.glob(f"{RUNS_DIR}/*/summary.json")):
    with open(summary_path) as f:
        summary = json.load(f)
    run_id = summary_path.split("/")[-2]
    print(f"Run: {run_id}")
    print(f"  Total jobs: {summary.get('total_jobs', '?')}")
    print(f"  Counts: {summary.get('counts', {})}")
    print()

Run: 20260305-063935-71c660a2
  Total jobs: 4
  Counts: {'ok': 4, 'failed': 0, 'skipped': 0}

Run: 20260305-064200-ba7c04cb
  Total jobs: 2
  Counts: {'ok': 2, 'failed': 0, 'skipped': 0}

Run: 20260305-071314-cc2fe733
  Total jobs: 4
  Counts: {'ok': 4, 'failed': 0, 'skipped': 0}

Run: 20260305-071546-cc2fe733
  Total jobs: 4
  Counts: {'ok': 4, 'failed': 0, 'skipped': 0}



In [ ]:
# Load API keys and model from Colab secrets into env (safe: no hard failure if missing)
import os

try:
    from google.colab import userdata
except Exception:
    userdata = None


def _safe_secret(name: str) -> str:
    if userdata is None:
        return ""
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

# Keep existing env value first; fallback to secret if present
API_KEY = os.environ.get("OPENROUTER_API_KEY") or _safe_secret("OPENROUTER_API_KEY")
MODEL = os.environ.get("TRACE_LITELLM_MODEL") or _safe_secret("TRACE_LITELLM_MODEL")

if API_KEY and MODEL:
    os.environ["OPENROUTER_API_KEY"] = API_KEY
    os.environ["OPENAI_API_KEY"] = API_KEY
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
    os.environ["TRACE_LITELLM_MODEL"] = MODEL
    print(f"Real mode ready: model={MODEL}")
elif API_KEY and not MODEL:
    print("WARNING: API key found but TRACE_LITELLM_MODEL not set.")
    print("         Set TRACE_LITELLM_MODEL in Colab Secrets or env to enable real mode.")
else:
    print("[info] No API key found. Real-mode runs may fail; stub mode still works.")

## Launch Gradio UI

The UI has 3 tabs:
1. **Launch Run** ? discover tasks/trainers dynamically, edit configs in YAML editor, choose provider (`custom/openai/openrouter`), and run with overrides including logger override (`default/none/<logger>`).
2. **Browse Runs** ? select a run, view results/config/summary, filter by suite/status/trainer, resume a run.
3. **Job Inspector** ? drill into individual jobs, view meta/events/state artifacts/TensorBoard dir.

The `--share` flag generates a public URL (auto-detected on Colab).

**Note:** This cell blocks while the UI is running. Open the printed URL to interact.


In [ ]:
!PYTHONPATH=/content/OpenTrace:/content/Trace-Bench/src:$PYTHONPATH python -m trace_bench ui --runs-dir "$RUNS_DIR" --share


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://f9f1596fdf4957b63e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
README.md: 100% 150/150 [00:00<00:00, 378kB/s]
py_src_index.json: 2.04MB [00:00, 136MB/s]
Generating train split: 140 examples [00:00, 2097.99 examples/s]


## TensorBoard

Each job stores TensorBoard event files under `jobs/<job_id>/tb/`. When the trainer uses `TensorboardLogger`, the runner auto-injects the per-job logdir.

To view:
```bash
tensorboard --logdir "$RUNS_DIR/<run_id>/jobs"
```

The Gradio UI also has a "Launch TensorBoard" button in the Browse tab.

## MLflow (optional)

MLflow mirroring is **opt-in**. Set `MLFLOW_TRACKING_URI` to enable:
```bash
export MLFLOW_TRACKING_URI=mlruns
python -m trace_bench run --config configs/m3_ui_demo.yaml --runs-dir runs
```

When enabled, the runner mirrors run params, job metrics, and artifact links to MLflow. The filesystem remains the canonical source of truth.

**Note:** Deep MLflow/OTel logger integration depends on the Trace team's upcoming PR. Current behavior is minimal mirroring of `score_initial`, `score_final`, `score_best`, and `time_seconds`.

## Summary

**Features demonstrated:**
- Gradio UI with 3 tabs: Launch, Browse, Job Inspector
- Dynamic task/trainer discovery (non-hardcoded)
- Provider clarity: `custom/openai/openrouter` selector + base URL/key/model fields
- Config editor: load, edit, save, run YAML configs from the UI
- Logger override from UI (`default/none/<logger class>`)
- Resume a previous run preserving the original run_id
- TensorBoard action returns URL/command guidance
- MLflow opt-in mirroring (filesystem canonical)
- CLI: `trace-bench run --logger ...` and `trace-bench ui --runs-dir ... --share --tasks-root ... --port ...`